In [0]:
# Databricks notebook: 04_map_transactions
# Purpose: Transform PhonePe map transaction JSON into an
# analytics-ready district/state-level Silver Delta table.

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    ArrayType,
    StringType,
    BooleanType,
    LongType,
    DoubleType
)

# ============================================================
# 1. CONFIGURATION
# ============================================================

BRONZE_TABLE = "workspace.bronze.phonepe_raw_json"
METADATA_TABLE = "workspace.silver.phonepe_file_metadata"
SILVER_TABLE = "workspace.silver.map_transactions"

# 34 country files + 1,224 state files
EXPECTED_SOURCE_FILES = 1_258

print("Configuration")
print(f"Bronze source : {BRONZE_TABLE}")
print(f"Metadata source: {METADATA_TABLE}")
print(f"Silver target : {SILVER_TABLE}")
print(f"Expected files: {EXPECTED_SOURCE_FILES:,}")


# ============================================================
# 2. SELECT MAP TRANSACTION SOURCE FILES
# ============================================================

bronze_df = spark.table(BRONZE_TABLE).alias("b")

map_transaction_metadata_df = (
    spark.table(METADATA_TABLE)
    .filter(
        (F.col("dataset_group") == "map") &
        (F.col("domain") == "transaction")
    )
    .alias("m")
)

map_transaction_source_df = (
    map_transaction_metadata_df
    .join(
        bronze_df,
        F.col("m.source_path") == F.col("b.source_path"),
        "inner"
    )
    .select(
        F.col("m.relative_path"),
        F.col("m.file_scope"),
        F.col("m.state_slug"),
        F.col("m.year"),
        F.col("m.quarter"),
        F.col("m.period"),
        F.col("b.raw_json"),
        F.col("b.source_path"),
        F.col("b.source_commit"),
        F.col("b.content_sha256"),
        F.col("b.ingested_at_utc")
    )
)

actual_source_files = (
    map_transaction_source_df
    .select("source_path")
    .distinct()
    .count()
)

print("\nSource validation")
print(f"Expected source files: {EXPECTED_SOURCE_FILES:,}")
print(f"Actual source files  : {actual_source_files:,}")

assert actual_source_files == EXPECTED_SOURCE_FILES, (
    f"Expected {EXPECTED_SOURCE_FILES:,} files, "
    f"but found {actual_source_files:,}."
)

print("Map transaction source validation passed.")


# ============================================================
# 3. DEFINE EXPLICIT JSON SCHEMA
# ============================================================

map_metric_schema = StructType([
    StructField("type", StringType(), True),
    StructField("count", LongType(), True),
    StructField("amount", DoubleType(), True)
])

map_area_schema = StructType([
    StructField("name", StringType(), True),
    StructField(
        "metric",
        ArrayType(map_metric_schema),
        True
    )
])

map_transaction_schema = StructType([
    StructField("success", BooleanType(), True),
    StructField("code", StringType(), True),

    StructField(
        "data",
        StructType([
            StructField(
                "hoverDataList",
                ArrayType(map_area_schema),
                True
            )
        ]),
        True
    ),

    StructField("responseTimestamp", LongType(), True)
])

print("\nExplicit map transaction schema created.")


# ============================================================
# 4. PARSE JSON
# ============================================================

parsed_map_transaction_df = (
    map_transaction_source_df
    .withColumn(
        "parsed_json",
        F.from_json(
            F.col("raw_json"),
            map_transaction_schema
        )
    )
)

parse_summary = (
    parsed_map_transaction_df
    .agg(
        F.count("*").alias("source_files"),

        F.sum(
            F.when(
                F.col("parsed_json.data.hoverDataList").isNull(),
                1
            ).otherwise(0)
        ).alias("files_without_map_data")
    )
    .first()
)

print("\nJSON parsing")
print(f"Source files          : {parse_summary['source_files']:,}")
print(
    f"Files without map data: "
    f"{parse_summary['files_without_map_data']:,}"
)

assert parse_summary["source_files"] == EXPECTED_SOURCE_FILES

assert parse_summary["files_without_map_data"] == 0, (
    "One or more files could not be parsed or do not "
    "contain hoverDataList."
)

print("Map transaction JSON parsing passed.")


# ============================================================
# 5. FLATTEN NESTED MAP RECORDS
# ============================================================

map_transactions_with_amount_df = (
    parsed_map_transaction_df

    .withColumn(
        "area_record",
        F.explode(
            F.col("parsed_json.data.hoverDataList")
        )
    )

    .withColumn(
        "metric_record",
        F.explode(
            F.col("area_record.metric")
        )
    )

    .select(
        F.col("file_scope"),

        F.when(
            F.col("file_scope") == "country",
            F.lit("state")
        )
        .otherwise(F.lit("district"))
        .alias("geography_level"),

        F.when(
            F.col("file_scope") == "state",
            F.col("state_slug")
        )
        .otherwise(F.lit(None).cast("string"))
        .alias("parent_state_slug"),

        F.trim(
            F.col("area_record.name")
        ).alias("area_name"),

        F.lower(
            F.regexp_replace(
                F.trim(F.col("area_record.name")),
                "[^a-zA-Z0-9]+",
                "-"
            )
        ).alias("area_slug"),

        F.col("year").cast("int").alias("year"),
        F.col("quarter").cast("int").alias("quarter"),
        F.col("period"),

        F.trim(
            F.col("metric_record.type")
        ).alias("payment_instrument_type"),

        F.col("metric_record.count")
            .cast("long")
            .alias("transaction_count"),

        F.col("metric_record.amount")
            .cast("double")
            .alias("transaction_amount"),

        F.col("parsed_json.responseTimestamp")
            .alias("response_timestamp_epoch_ms"),

        F.col("relative_path"),
        F.col("source_path"),
        F.col("source_commit"),
        F.col("content_sha256"),
        F.col("ingested_at_utc")
    )
)

print("\nNested map transaction records flattened.")


# ============================================================
# 6. HANDLE OPTIONAL TRANSACTION AMOUNT
# ============================================================

amount_non_null_rows = (
    map_transactions_with_amount_df
    .filter(F.col("transaction_amount").isNotNull())
    .count()
)

if amount_non_null_rows == 0:
    map_transactions_final_df = (
        map_transactions_with_amount_df
        .drop("transaction_amount")
    )

    print(
        "transaction_amount removed because the field is "
        "absent from every map transaction record."
    )
else:
    map_transactions_final_df = (
        map_transactions_with_amount_df
    )

    print(
        f"transaction_amount retained: "
        f"{amount_non_null_rows:,} populated records."
    )


# ============================================================
# 7. DATA-QUALITY VALIDATION
# ============================================================

quality_result = (
    map_transactions_final_df
    .agg(
        F.count("*").alias("total_rows"),

        F.sum(
            F.when(
                F.col("area_name").isNull() |
                F.col("geography_level").isNull() |
                F.col("payment_instrument_type").isNull() |
                F.col("transaction_count").isNull(),
                1
            ).otherwise(0)
        ).alias("missing_required_values"),

        F.sum(
            F.when(
                F.col("transaction_count") < 0,
                1
            ).otherwise(0)
        ).alias("negative_transaction_counts"),

        F.countDistinct("source_path")
            .alias("represented_source_files")
    )
    .first()
)

duplicate_business_keys = (
    map_transactions_final_df
    .groupBy(
        "source_path",
        "area_name",
        "payment_instrument_type"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("\nData-quality results")
print(f"Total Silver rows           : {quality_result['total_rows']:,}")
print(
    f"Missing required values     : "
    f"{quality_result['missing_required_values']:,}"
)
print(
    f"Negative transaction counts : "
    f"{quality_result['negative_transaction_counts']:,}"
)
print(
    f"Source files represented    : "
    f"{quality_result['represented_source_files']:,}"
)
print(f"Duplicate business keys     : {duplicate_business_keys:,}")

assert quality_result["total_rows"] > 0
assert quality_result["missing_required_values"] == 0
assert quality_result["negative_transaction_counts"] == 0
assert quality_result["represented_source_files"] == EXPECTED_SOURCE_FILES
assert duplicate_business_keys == 0

print("Map transaction quality checks passed.")


# ============================================================
# 8. SAVE MANAGED DELTA TABLE
# ============================================================

(
    map_transactions_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"\nSilver Delta table created: {SILVER_TABLE}")


# ============================================================
# 9. VALIDATE SAVED TABLE
# ============================================================

saved_map_transactions_df = spark.table(SILVER_TABLE)

saved_summary = (
    saved_map_transactions_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("source_path").alias("source_file_count"),
        F.min("period").alias("first_period"),
        F.max("period").alias("last_period"),
        F.sum("transaction_count").alias("total_transaction_count")
    )
    .first()
)

print("\nSaved-table validation")
print(f"Saved rows              : {saved_summary['row_count']:,}")
print(f"Source files represented: {saved_summary['source_file_count']:,}")
print(f"First period            : {saved_summary['first_period']}")
print(f"Last period             : {saved_summary['last_period']}")
print(
    f"Total transaction count : "
    f"{saved_summary['total_transaction_count']:,}"
)

assert saved_summary["row_count"] > 0
assert saved_summary["source_file_count"] == EXPECTED_SOURCE_FILES
assert saved_summary["first_period"] == "2018-Q1"
assert saved_summary["last_period"] == "2026-Q2"

print("Final map transaction Delta-table validation passed.")


# ============================================================
# 10. DISPLAY SAMPLE
# ============================================================

display(
    saved_map_transactions_df
    .select(
        "geography_level",
        "parent_state_slug",
        "area_name",
        "year",
        "quarter",
        "period",
        "payment_instrument_type",
        "transaction_count"
    )
    .orderBy(
        "year",
        "quarter",
        "geography_level",
        "parent_state_slug",
        "area_name"
    )
    .limit(20)
)

Configuration
Bronze source : workspace.bronze.phonepe_raw_json
Metadata source: workspace.silver.phonepe_file_metadata
Silver target : workspace.silver.map_transactions
Expected files: 1,258

Source validation
Expected source files: 1,258
Actual source files  : 1,258
Map transaction source validation passed.

Explicit map transaction schema created.

JSON parsing
Source files          : 1,258
Files without map data: 0
Map transaction JSON parsing passed.

Nested map transaction records flattened.
transaction_amount retained: 27,840 populated records.

Data-quality results
Total Silver rows           : 27,840
Missing required values     : 0
Negative transaction counts : 0
Source files represented    : 1,258
Duplicate business keys     : 0
Map transaction quality checks passed.

Silver Delta table created: workspace.silver.map_transactions

Saved-table validation
Saved rows              : 27,840
Source files represented: 1,258
First period            : 2018-Q1
Last period             : 

geography_level,parent_state_slug,area_name,year,quarter,period,payment_instrument_type,transaction_count
district,andaman-&-nicobar-islands,nicobar district,2018,1,2018-Q1,TOTAL,696
district,andaman-&-nicobar-islands,north and middle andaman district,2018,1,2018-Q1,TOTAL,537
district,andaman-&-nicobar-islands,south andaman district,2018,1,2018-Q1,TOTAL,7856
district,andhra-pradesh,alluri sitharama raju district,2018,1,2018-Q1,TOTAL,38878
district,andhra-pradesh,anakapalli district,2018,1,2018-Q1,TOTAL,369336
district,andhra-pradesh,ananthapuramu district,2018,1,2018-Q1,TOTAL,519650
district,andhra-pradesh,annamayya district,2018,1,2018-Q1,TOTAL,272082
district,andhra-pradesh,bapatla district,2018,1,2018-Q1,TOTAL,229517
district,andhra-pradesh,chittoor district,2018,1,2018-Q1,TOTAL,305531
district,andhra-pradesh,dr br ambedkar konaseema district,2018,1,2018-Q1,TOTAL,261840


# Silver Map Transactions — Completion Summary

The `workspace.silver.map_transactions` Delta table was successfully created and validated.

## Table grain

Each record represents one geographical area, payment instrument, and quarter.

- Country-level source files produce state-level records.
- State-level source files produce district-level records.

## Available metrics

- Transaction count
- Transaction amount
- Payment instrument type
- State/district geography
- Year and quarter
- Complete source lineage

## Important source finding

PhonePe's aggregated transaction files provide category-level transaction counts but no transaction amounts.

The map transaction files retain transaction amounts. Therefore:

- Category analysis will use transaction counts.
- State and district monetary analysis will use map transaction amounts.
- Machine-learning models can forecast both regional transaction counts and regional transaction values.

## Quality controls

- Source-file count validation
- Explicit JSON schema
- Required-value validation
- Negative-value validation
- Duplicate business-key validation
- Period-coverage validation
- Source-lineage preservation